In [12]:
! pip install langchain langchain-community langchain-groq langchain-text-splitters
! pip install HuggingFaceEmbeddings
! pip install faiss-cpu


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement HuggingFaceEmbeddings (from versions: none)

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for HuggingFaceEmbeddings



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os

In [33]:
load_dotenv()
document_loader = PyPDFLoader("Gen AI.pdf")

document=document_loader.load()
print(len(document))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(document)

print(len(chunks))

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vector.as_retriever(search_kwargs={"k": 2})


llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="openai/gpt-oss-120b",
    temperature=0
)


question = ["what is gen ai?",
            "what is the full form of gen ai?",
            "what is rag?",
            "who is the PM of India?",
            "what is playright?",
            " what is the capital of India?"
        ]
for q in question:

    response = retriever.invoke(q)  
    context = "\n".join(
        doc.page_content for doc in response
        )      
    prompt = f"""
You are a question-answering assistant for a document.

IMPORTANT RULES:
1. Answer ONLY using the provided context.
2. Do not use your own general knowledge.
3. Do not make assumptions.
4. If the answer is not explicitly available in the context,
   answer exactly: "I don't know...."

Context:
{context}

Question:
{q}

Answer:
"""
    result = llm.invoke(prompt)
    print(f"Question: {q}")
    print(f"Answer: {result.content}")

# print(document[0].page_content)
# print(document[0].metadata)
# print(len(document))



35
40


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2682.90it/s]


Question: what is gen ai?
Answer: Generative AI (GenAI) is a type of artificial intelligence that can create new content—such as text, images, audio, video, or code—based on the instructions given by a user.
Question: what is the full form of gen ai?
Answer: Generative AI.
Question: what is rag?
Answer: RAG (Retrieval‑Augmented Generation) is a technique in which an AI model first retrieves relevant information—such as from company documents—and then uses that retrieved content to generate its answer.
Question: who is the PM of India?
Answer: I don't know....
Question: what is playright?
Answer: I don't know....
Question:  what is the capital of India?
Answer: New Delhi
